[Open in Colab](https://colab.research.google.com/github/owner/repo/blob/main/notebooks/quanot_benchmark.ipynb)

In [ ]:
# Colab setup
import sys
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib", "-q"])

# Quanot Performance Benchmark

Benchmarking the quantum-inspired reservoir computing system.

## Metrics
- **Latency**: Time per `process()` call (us)
- **Throughput**: Messages/second at various batch sizes
- **Memory**: State vector allocation overhead
- **SIMD Potential**: Floating-point operation density
- **Training**: Ridge regression parallelization speedup

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import statistics

## 1. Simulated Quanot Benchmark

Since we're benchmarking the Rust implementation, we'll create a Python mirror
of the reservoir computing logic to establish baseline metrics.

In [ ]:
class ReservoirBenchmark:
    """Python mirror of quanot reservoir for baseline metrics."""
    
    def __init__(self, input_dim=128, reservoir_size=1000):
        self.input_dim = input_dim
        self.reservoir_size = reservoir_size
        
        # Initialize weights (matches Rust implementation)
        np.random.seed(42)
        self.w_in = np.random.randn(reservoir_size, input_dim) * 0.1
        
        # Sparse reservoir weights
        self.w = np.random.randn(reservoir_size, reservoir_size) * 0.01
        self.w[np.random.rand(reservoir_size, reservoir_size) > 0.01] = 0
        
        # Scale spectral radius to ~0.95
        eigenvalues = np.linalg.eigvals(self.w)
        spectral_radius = np.max(np.abs(eigenvalues))
        self.w = self.w * (0.95 / spectral_radius)
        
        self.state = np.zeros(reservoir_size)
        self.leak_rate = 0.3
        self.noise_level = 0.001
        self.state_history = []
    
    def step(self, input_vec):
        """One reservoir step (matches Rust reservoir.step)"""
        pre_activation = self.w_in @ input_vec + self.w @ self.state
        # TanH activation with leak
        new_state = (1 - self.leak_rate) * self.state + self.leak_rate * np.tanh(pre_activation)
        # Add noise
        new_state += np.random.randn(self.reservoir_size) * self.noise_level
        self.state = new_state
        return new_state.copy()
    
    def process(self, text):
        """Process text: encode + reservoir step"""
        # Simple encoding (matches Rust TextEncoder)
        encoded = np.random.randn(self.input_dim) * 0.1
        return self.step(encoded)
    
    def reset(self):
        self.state = np.zeros(self.reservoir_size)
        self.state_history = []

## 2. Latency Benchmarks

In [ ]:
def benchmark_latency(reservoir, n_iterations=1000, warmup=100):
    """Measure latency per process() call."""
    
    # Warmup
    for _ in range(warmup):
        reservoir.process("warmup")
    reservoir.reset()
    
    times = []
    for _ in range(n_iterations):
        start = time.perf_counter()
        reservoir.process("benchmark input")
        end = time.perf_counter()
        times.append((end - start) * 1e6)  # microseconds
    
    return {
        'mean': statistics.mean(times),
        'median': statistics.median(times),
        'stdev': statistics.stdev(times) if len(times) > 1 else 0,
        'min': min(times),
        'max': max(times),
        'p95': sorted(times)[int(len(times) * 0.95)],
        'p99': sorted(times)[int(len(times) * 0.99)]
    }


# Run latency benchmarks at different reservoir sizes
reservoir_sizes = [100, 500, 1000]
latency_results = {}

for size in reservoir_sizes:
    print(f"Benchmarking reservoir_size={size}...")
    r = ReservoirBenchmark(input_dim=128, reservoir_size=size)
    latency_results[size] = benchmark_latency(r)
    print(f"  mean={latency_results[size]['mean']:.2f}μs, p95={latency_results[size]['p95']:.2f}μs")

In [ ]:
# Plot latency results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sizes = list(latency_results.keys())
means = [latency_results[s]['mean'] for s in sizes]
p95s = [latency_results[s]['p95'] for s in sizes]

axes[0].plot(sizes, means, 'b-o', label='mean')
axes[0].plot(sizes, p95s, 'r--', label='p95')
axes[0].set_xlabel('Reservoir Size')
axes[0].set_ylabel('Latency (μs)')
axes[0].set_title('Process Latency vs Reservoir Size')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Log-log plot to see scaling
axes[1].loglog(sizes, means, 'b-o', label='mean')
axes[1].loglog(sizes, p95s, 'r--', label='p95')
axes[1].set_xlabel('Reservoir Size (log)')
axes[1].set_ylabel('Latency (μs, log)')
axes[1].set_title('Scaling Behavior (log-log)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Throughput Benchmarks

In [ ]:
def benchmark_throughput(reservoir, batch_sizes=[1, 10, 50, 100, 500], duration_ms=100):
    """Measure throughput at different batch sizes."""
    results = {}
    
    for batch_size in batch_sizes:
        # Warmup
        for _ in range(max(10, batch_size)):
            reservoir.process("warmup")
        reservoir.reset()
        
        start = time.perf_counter()
        count = 0
        elapsed = 0
        
        while elapsed < duration_ms / 1000:
            for _ in range(batch_size):
                reservoir.process(f"batch item {count}")
                count += 1
            elapsed = time.perf_counter() - start
        
        msgs_per_sec = count / elapsed
        results[batch_size] = {
            'messages': count,
            'duration_sec': elapsed,
            'msgs_per_sec': msgs_per_sec
        }
        print(f"  batch={batch_size}: {msgs_per_sec:.0f} msg/s")
    
    return results

print("Throughput benchmarks:")
r = ReservoirBenchmark(input_dim=128, reservoir_size=1000)
throughput_results = benchmark_throughput(r)

In [ ]:
# Plot throughput
batch_sizes = list(throughput_results.keys())
msgs_per_sec = [throughput_results[b]['msgs_per_sec'] for b in batch_sizes]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(batch_sizes, msgs_per_sec, color='steelblue')
axes[0].set_xlabel('Batch Size')
axes[0].set_ylabel('Messages/Second')
axes[0].set_title('Throughput vs Batch Size')
axes[0].grid(True, alpha=0.3)

# Efficiency (msgs/sec per batch item)
efficiency = [throughput_results[b]['msgs_per_sec'] / b for b in batch_sizes]
axes[1].plot(batch_sizes, efficiency, 'g-o')
axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('Efficiency (msg/s / item)')
axes[1].set_title('Scaling Efficiency')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Memory Allocation Analysis

In [ ]:
import sys

def estimate_memory_usage(reservoir_size, input_dim=128):
    """Estimate memory for a single reservoir state."""
    
    # State vector: f64 = 8 bytes
    state_bytes = reservoir_size * 8
    
    # w_in: reservoir_size * input_dim * 8 bytes
    w_in_bytes = reservoir_size * input_dim * 8
    
    # w (sparse ~1% density): reservoir_size^2 * 0.01 * 8 bytes
    w_sparse_bytes = reservoir_size * reservoir_size * 0.01 * 8
    
    # Full w for reference
    w_full_bytes = reservoir_size * reservoir_size * 8
    
    return {
        'state_vector_bytes': state_bytes,
        'w_in_bytes': w_in_bytes,
        'w_sparse_bytes': int(w_sparse_bytes),
        'w_full_bytes': w_full_bytes,
        'total_sparse_bytes': state_bytes + w_in_bytes + int(w_sparse_bytes)
    }

# Memory analysis for different reservoir sizes
memory_by_size = {}
for size in [100, 500, 1000, 2000]:
    memory_by_size[size] = estimate_memory_usage(size)
    m = memory_by_size[size]
    print(f"reservoir_size={size}:")
    print(f"  state vector: {m['state_vector_bytes']/1024:.1f} KB")
    print(f"  w_in: {m['w_in_bytes']/1024/1024:.1f} MB")
    print(f"  w (sparse): {m['w_sparse_bytes']/1024:.1f} KB")
    print(f"  total (sparse): {m['total_sparse_bytes']/1024/1024:.2f} MB")
    print(f"  w (dense): {m['w_full_bytes']/1024/1024:.1f} MB")

In [ ]:
# Visualize memory scaling
sizes = list(memory_by_size.keys())
total_sparse = [memory_by_size[s]['total_sparse_bytes']/1024/1024 for s in sizes]
w_full = [memory_by_size[s]['w_full_bytes']/1024/1024 for s in sizes]

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(sizes, total_sparse, 'b-o', label='Sparse (actual)')
ax.loglog(sizes, w_full, 'r--', label='Dense (if not sparse)')
ax.set_xlabel('Reservoir Size (log)')
ax.set_ylabel('Memory (MB, log)')
ax.set_title('Memory Scaling: Sparse vs Dense')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. SIMD Vectorization Analysis

In [ ]:
def count_flops(reservoir_size, input_dim):
    """Count floating-point operations per step."""
    
    # w_in @ input_vec: reservoir_size * input_dim multiplications + additions
    w_in_flops = reservoir_size * input_dim * 2  # mul + add per element
    
    # w @ state: reservoir_size * reservoir_size * 0.01 (sparse) * 2
    w_sparse_flops = int(reservoir_size * reservoir_size * 0.01) * 2
    
    # tanh: reservoir_size operations
    tanh_flops = reservoir_size * 5  # approximation
    
    # leak update: reservoir_size * 3
    leak_flops = reservoir_size * 3
    
    return {
        'w_in_flops': w_in_flops,
        'w_sparse_flops': w_sparse_flops,
        'tanh_flops': tanh_flops,
        'leak_flops': leak_flops,
        'total_flops': w_in_flops + w_sparse_flops + tanh_flops + leak_flops
    }

# FLOP analysis
print("FLOPs per reservoir step (reservoir_size=1000, input_dim=128):")
flops = count_flops(1000, 128)
for k, v in flops.items():
    print(f"  {k}: {v:,}")

In [ ]:
# SIMD efficiency: what vector width would we need?
print("\nSIMD Vectorization Opportunities:")
print("="*50)
for size in [500, 1000, 2000]:
    flops = count_flops(size, 128)
    total = flops['total_flops']
    
    # AVX-256 is 256 bits = 4 f64s per vector register
    avx256_width = 4
    avx512_width = 8
    
    avx256_ops = total / avx256_width
    avx512_ops = total / avx512_width
    
    print(f"\nReservoir size {size}:")
    print(f"  Total FLOPs: {total:,}")
    print(f"  AVX-256 vector ops needed: {avx256_ops:.0f}")
    print(f"  AVX-512 vector ops needed: {avx512_ops:.0f}")
    print(f"  Vectorization efficiency: {avx256_ops/total*100:.1f}% theoretical max")

## 6. Training Parallelization Analysis

In [ ]:
def simulate_ridge_regression_training(n_samples, n_features, n_workers=4):
    """Simulate ridge regression training time scaling."""
    
    # Create test data
    X = np.random.randn(n_samples, n_features)
    y = np.random.randn(n_samples)
    
    # Serial ridge regression via SVD
    start = time.perf_counter()
    # Ridge regression: (X^T X + λI)^-1 X^T y
    lambda_reg = 0.01
    XtX = X.T @ X + lambda_reg * np.eye(n_features)
    Xty = X.T @ y
    # Solve using Cholesky (faster than SVD for ridge)
    coeffs = np.linalg.solve(XtX, Xty)
    serial_time = time.perf_counter() - start
    
    # Parallel: split samples across workers
    start = time.perf_counter()
    chunk_size = n_samples // n_workers
    results = []
    for i in range(n_workers):
        start_idx = i * chunk_size
        end_idx = start_idx + chunk_size if i < n_workers - 1 else n_samples
        X_chunk = X[start_idx:end_idx]
        y_chunk = y[start_idx:end_idx]
        # Local computation
        XtX_chunk = X_chunk.T @ X_chunk + lambda_reg * np.eye(n_features) / n_workers
        Xty_chunk = X_chunk.T @ y_chunk
        results.append((XtX_chunk, Xty_chunk))
    
    # Aggregate results
    XtX_total = sum(r[0] for r in results)
    Xty_total = sum(r[1] for r in results)
    coeffs_parallel = np.linalg.solve(XtX_total, Xty_total)
    parallel_time = time.perf_counter() - start
    
    return {
        'serial_time': serial_time * 1000,  # ms
        'parallel_time': parallel_time * 1000,
        'speedup': serial_time / parallel_time if parallel_time > 0 else 1
    }

print("Ridge Regression Training Scaling:")
print("="*50)
for n_samples in [100, 1000, 5000]:
    for n_features in [128, 512]:
        result = simulate_ridge_regression_training(n_samples, n_features)
        print(f"samples={n_samples}, features={n_features}: "
              f"serial={result['serial_time']:.2f}ms, "
              f"parallel={result['parallel_time']:.2f}ms, "
              f"speedup={result['speedup']:.2f}x")

## 7. Summary & Optimization Recommendations

In [ ]:
print("\n" + "="*60)
print("QUANOT BENCHMARK SUMMARY")
print("="*60)

print("\n[L] LATENCY (reservoir_size=1000):")
r1000 = latency_results[1000]
print(f"   Mean: {r1000['mean']:.2f} μs")
print(f"   P95:   {r1000['p95']:.2f} μs")
print(f"   P99:   {r1000['p99']:.2f} μs")

print("\n[T] THROUGHPUT:")
print(f"   Batch=1:  {throughput_results[1]['msgs_per_sec']:.0f} msg/s")
print(f"   Batch=10: {throughput_results[10]['msgs_per_sec']:.0f} msg/s")
print(f"   Batch=50: {throughput_results[50]['msgs_per_sec']:.0f} msg/s")

print("\n[M] MEMORY (reservoir_size=1000):")
m = estimate_memory_usage(1000)
print(f"   Total sparse: {m['total_sparse_bytes']/1024/1024:.2f} MB")
print(f"   Dense equivalent: {m['w_full_bytes']/1024/1024:.1f} MB")

print("\n[S] SIMD OPPORTUNITIES:")
flops = count_flops(1000, 128)
print(f"   Total FLOPs/step: {flops['total_flops']:,}")
print(f"   Vectorizable: {(flops['w_in_flops'] + flops['w_sparse_flops'])/4:.0f} AVX-256 ops")

print("\n[O] OPTIMIZATION RECOMMENDATIONS:")
print("   1. Memory pool: Pre-allocate state vectors, reuse instead of alloc")
print("   2. SIMD: Vectorize w_in and w matrix multiplications")
print("   3. Parallel training: Split ridge regression across cores for large datasets")
print("   4. Sparse ops: Use sparse BLAS for reservoir matrix multiply")

In [ ]:
# Save benchmark data for Rust comparison
benchmark_data = {
    'latency': latency_results,
    'throughput': throughput_results,
    'memory': {k: {kk: vv for kk, vv in v.items()} for k, v in memory_by_size.items()},
    'flops': {k: count_flops(k, 128) for k in [500, 1000, 2000]}
}

import json
with open('quanot_benchmark_data.json', 'w') as f:
    json.dump(benchmark_data, f, indent=2)
print("Benchmark data saved to quanot_benchmark_data.json")

## 8. Compare with Rust Implementation

To benchmark the actual Rust implementation, add a benchmark binary:

```rust
// src/bin/quanot_bench.rs
use star::quanot::Quanot;
use std::time::Instant;

fn main() {
    let mut q = Quanot::new(128, 1000);
    
    // Warmup
    for _ in 0..100 { q.process("warmup"); }
    
    // Benchmark
    let start = Instant::now();
    for i in 0..10000 {
        q.process(&format!("bench {}", i));
    }
    let elapsed = start.elapsed();
    
    println!("{\"latency_ms\":{:.3}}", elapsed.as_secs_f64() * 1000.0 / 10000.0);
}
```

Run with: `cargo bench --bin quanot_bench`